In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix
)

print("TensorFlow version:", tf.__version__)
print("Notebook environment is ready!")


TensorFlow version: 2.21.0
Notebook environment is ready!


In [3]:
# Reproducibility and common experiment settings
SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 5

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]

print("Random seed:", SEED)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Number of classes:", NUM_CLASSES)
print("Classes:", CLASS_NAMES)

Random seed: 42
Image size: (224, 224)
Batch size: 32
Number of classes: 5
Classes: ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']


In [4]:
from pathlib import Path

# Find the project root automatically
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "aptos2019-blindness-detection"
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMAGES_DIR = DATA_DIR / "train_images"

print("Project root:", PROJECT_ROOT)
print("Dataset folder exists:", DATA_DIR.exists())
print("train.csv exists:", TRAIN_CSV.exists())
print("train_images folder exists:", TRAIN_IMAGES_DIR.exists())
print("Number of training images:", len(list(TRAIN_IMAGES_DIR.glob("*.png"))))

Project root: c:\Users\MSI\Desktop\DL-Blindness-ditect\Deep-Learning-Assignment-Group-ID-5
Dataset folder exists: True
train.csv exists: True
train_images folder exists: True
Number of training images: 3662


In [5]:
# Load the APTOS training labels
df = pd.read_csv(TRAIN_CSV)

print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

print("\nClass distribution:")
print(df["diagnosis"].value_counts().sort_index())

df.head()

Dataset shape: (3662, 2)

Column names: ['id_code', 'diagnosis']

Missing values:
id_code      0
diagnosis    0
dtype: int64

Class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64


,id_code,diagnosis
0,000c1434d8d7,2
1,001639a390f0,4
2,0024cdab0c1e,1
3,002c21358ce6,0
4,005b95c28852,0


In [6]:
# Add the complete image path for every row
df["image_path"] = df["id_code"].apply(
    lambda image_id: str(TRAIN_IMAGES_DIR / f"{image_id}.png")
)

# Check whether every referenced image exists
df["image_exists"] = df["image_path"].apply(os.path.exists)

print("Total CSV records:", len(df))
print("Images found:", df["image_exists"].sum())
print("Missing images:", (~df["image_exists"]).sum())

df[["id_code", "diagnosis", "image_path"]].head()

Total CSV records: 3662
Images found: 3662
Missing images: 0


,id_code,diagnosis,image_path
0,000c1434d8d7,2,c:\Users\MSI\Desktop\DL-Blindness-ditect\Deep-...
1,001639a390f0,4,c:\Users\MSI\Desktop\DL-Blindness-ditect\Deep-...
2,0024cdab0c1e,1,c:\Users\MSI\Desktop\DL-Blindness-ditect\Deep-...
3,002c21358ce6,0,c:\Users\MSI\Desktop\DL-Blindness-ditect\Deep-...
4,005b95c28852,0,c:\Users\MSI\Desktop\DL-Blindness-ditect\Deep-...
